# Generate CodeBLEU averages across all mutation tiers 

In [ ]:
import sys
import json
import pandas as pd
from pathlib import Path

# Add project root to Python path
sys.path.append(str(Path("..").resolve()))

from src.evaluation.CodeBLEU.calc_code_bleu import calc_dataset_average_codebleu

print("Environment setup complete.")

Environment setup complete.


In [ ]:

BASE_DIR = Path("transformations")
output_dir = Path("output")
output_dir.mkdir(parents=True, exist_ok=True)
datasets = ["droidcollection", "codet_m4"]
tiers = ["tier_1", "tier_2", "tier_3", "tier_4"]

all_results = {}

# Loop through every dataset and its mutation tiers
for dataset in datasets:
    print(f"\n{'='*40}\nProcessing Dataset: {dataset}\n{'='*40}")
    all_results[dataset] = {}
    
    for tier in tiers:
        # Target the exact parquet files in your subdirectories
        file_path = BASE_DIR / dataset / tier / "augmented_dataset.parquet"
        
        if not file_path.exists():
            print(f"Skipping: {file_path} (File not found)")
            continue
            
        print(f"--> Loading and evaluating {tier}...")

        df = pd.read_parquet(file_path)
        
        if "original_code" not in df.columns or "mutated_code" not in df.columns:
            print(f"Error: Missing columns in {file_path}. Found: {df.columns.tolist()}")
            continue

        references = df["original_code"].astype(str).tolist()
        predictions = df["mutated_code"].astype(str).tolist()

        tier_averages = calc_dataset_average_codebleu(
            references=references,
            predictions=predictions,
            lang="python"
        )
        
        all_results[dataset][tier] = tier_averages

print("\nAll evaluations completed successfully!")

Show and save results:

In [ ]:
output_dir = Path("output")
output_dir.mkdir(parents=True, exist_ok=True)

# Process, display, and save the metrics for each dataset
for dataset in datasets:
    if dataset in all_results and all_results[dataset]:
        print(f"\n### {dataset.upper()} BENCHMARK MATRIX ###")
        
        # Convert the tier metrics to a structured DataFrame
        df_view = pd.DataFrame(all_results[dataset]).T
        
        # Order columns cleanly with the final composite score first
        column_order = [
            "codebleu", 
            "ngram_match_score", 
            "weighted_ngram_match_score", 
            "syntax_match_score", 
            "dataflow_match_score"
        ]
        df_view = df_view[column_order]
        
        # Render the table inside the notebook UI
        display(df_view)
        
        # Save to a clean CSV file inside your output/ folder
        csv_filename = output_dir / f"{dataset}_codebleu_results.csv"
        df_view.to_csv(csv_filename, index_label="tier")
        print(f"Saved CSV metrics to: {csv_filename}")

# Save a global JSON file of all data points as a backup
json_filename = output_dir / "global_codebleu_results.json"
with open(json_filename, "w", encoding="utf-8") as f:
    json.dump(all_results, f, indent=4)

print(f"\n Global JSON backup saved successfully to: {json_filename}")